# 05 — Model Comparison & Calibration (Give Me Some Credit)

Consolidates what we worked through live: racing **XGBoost vs LightGBM vs CatBoost**, tuning the
laggard, and **calibrating** probabilities so they are honest, not just well-ranked.

Runs on the same cleaned Give Me Some Credit data as notebook 04.

## 1. Setup, load, clean (same as notebook 04)

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from catboost import CatBoostClassifier

TARGET = "SeriousDlqin2yrs"
pastdue = ["NumberOfTime30-59DaysPastDueNotWorse","NumberOfTime60-89DaysPastDueNotWorse","NumberOfTimes90DaysLate"]

def clean(df):
    df = df.copy()
    df.loc[df["age"] == 0, "age"] = np.nan
    for c in pastdue:
        df.loc[df[c].isin([96, 98]), c] = np.nan
    df["MonthlyIncome_missing"] = df["MonthlyIncome"].isna().astype(int)
    return df

data = clean(pd.read_csv("../data/raw/cs-training.csv", index_col=0))
X, y = data.drop(columns=[TARGET]), data[TARGET]
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=.2, stratify=y, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_tmp, y_tmp, test_size=.25, stratify=y_tmp, random_state=42)
spw = (y_tr == 0).sum() / (y_tr == 1).sum()
print("split done. scale_pos_weight =", round(spw, 2))

split done. scale_pos_weight = 13.96


## 2. Race three gradient-boosting libraries

Same data, same imbalance weighting, each with early stopping. In practice you try all three and keep
the winner — no library is automatically best on your data.

In [2]:
def report(model, label, X_te=X_test, y_te=y_test):
    p = model.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, p)
    ks = ks_2samp(p[y_te == 1], p[y_te == 0]).statistic
    print(f"  {label:16s} AUC={auc:.4f}  Gini={2*auc-1:.4f}  KS={ks:.4f}")
    return p

xgb = XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=4, subsample=0.8,
    colsample_bytree=0.8, scale_pos_weight=spw, eval_metric="auc",
    early_stopping_rounds=50, n_jobs=-1, random_state=42)
xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

lgb = LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=31, subsample=0.8,
    colsample_bytree=0.8, scale_pos_weight=spw, random_state=42, n_jobs=-1, verbose=-1)
lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="auc",
    callbacks=[early_stopping(50, verbose=False), log_evaluation(0)])

cat = CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=4, scale_pos_weight=spw,
    eval_metric="AUC", early_stopping_rounds=50, random_seed=42, verbose=False)
cat.fit(X_tr, y_tr, eval_set=(X_val, y_val))

print("Test results:")
report(xgb, "XGBoost (untuned)")
report(lgb, "LightGBM (untuned)")
report(cat, "CatBoost (untuned)")

Test results:
  XGBoost (untuned) AUC=0.8686  Gini=0.7372  KS=0.5854
  LightGBM (untuned) AUC=0.8411  Gini=0.6821  KS=0.5400
  CatBoost (untuned) AUC=0.8689  Gini=0.7378  KS=0.5819


array([0.10817114, 0.23719955, 0.07100698, ..., 0.16581096, 0.06767154,
       0.63460554], shape=(30000,))

LightGBM lands behind here **only because its settings were borrowed from XGBoost**. Its growth is
leaf-wise; its key brakes are `num_leaves` (fewer = simpler) and `min_child_samples` (higher = more
regularized). Defaults are not destiny — tune it on its own terms.

## 3. Tune LightGBM with Optuna (watch it catch up)

In [3]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = dict(
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        feature_fraction=trial.suggest_float("feature_fraction", 0.5, 1.0),
        bagging_fraction=trial.suggest_float("bagging_fraction", 0.5, 1.0),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
    )
    m = LGBMClassifier(n_estimators=1000, scale_pos_weight=spw, random_state=42,
        n_jobs=-1, verbose=-1, bagging_freq=1, **params)
    m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="auc",
        callbacks=[early_stopping(50, verbose=False), log_evaluation(0)])
    return roc_auc_score(y_val, m.predict_proba(X_val)[:, 1])

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=25, show_progress_bar=False)

lgb_tuned = LGBMClassifier(n_estimators=1000, scale_pos_weight=spw, random_state=42,
    n_jobs=-1, verbose=-1, bagging_freq=1, **study.best_params)
lgb_tuned.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric="auc",
    callbacks=[early_stopping(50, verbose=False), log_evaluation(0)])
print("best params:", study.best_params)
print("\nTest results after tuning LightGBM:")
report(lgb, "LightGBM (untuned)")
report(lgb_tuned, "LightGBM (tuned)")
report(xgb, "XGBoost")
report(cat, "CatBoost")

best params: {'num_leaves': 40, 'min_child_samples': 181, 'learning_rate': 0.013098203069180081, 'feature_fraction': 0.5924098885903923, 'bagging_fraction': 0.8098231010179132, 'reg_lambda': 2.5783197511365343}

Test results after tuning LightGBM:
  LightGBM (untuned) AUC=0.8411  Gini=0.6821  KS=0.5400
  LightGBM (tuned) AUC=0.8629  Gini=0.7258  KS=0.5701
  XGBoost          AUC=0.8686  Gini=0.7372  KS=0.5854
  CatBoost         AUC=0.8689  Gini=0.7378  KS=0.5819


array([0.10817114, 0.23719955, 0.07100698, ..., 0.16581096, 0.06767154,
       0.63460554], shape=(30000,))

## 4. Calibration: are the probabilities honest?

Boosting models **rank** well (good AUC) but their raw probabilities can be wrong. `scale_pos_weight`
(used to fight imbalance) inflates every predicted PD. Check it: bin borrowers by predicted PD and
compare what the model **says** to what **really happened**.

In [4]:
def reliability(p, y_true=y_test, n=10):
    d = pd.DataFrame({"p": p, "y": y_true.values})
    d["bucket"] = pd.qcut(d["p"], n, labels=False, duplicates="drop")
    return (d.groupby("bucket").agg(n=("y","size"), model_says=("p","mean"),
            really_happened=("y","mean")) * [1, 100, 100]).round(1)

p_raw = xgb.predict_proba(X_test)[:, 1]
print("XGBoost RAW probabilities vs reality (model_says & really_happened are %):\n")
print(reliability(p_raw).to_string())

XGBoost RAW probabilities vs reality (model_says & really_happened are %):

           n  model_says  really_happened
bucket                                   
0       3000         6.2              0.5
1       3000         9.3              0.3
2       3000        12.2              1.0
3       3000        15.4              1.1
4       3000        19.7              1.8
5       3000        26.3              2.6
6       3000        36.2              3.6
7       3000        48.4              6.7
8       3000        62.3             12.1
9       3000        85.8             37.1


The `model_says` column is 2-10x higher than `really_happened`: the model cries wolf. AUC never
notices, because the *ranking* is fine — which is exactly why calibration must be checked separately.

## 5. Fix it: Platt (sigmoid) and isotonic calibration

Fit a small correction model on a **holdout** (the validation set) and apply to test. Platt = smooth
S-curve (safer on little data); isotonic = free-form monotonic (best with lots of data).

In [5]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

platt = CalibratedClassifierCV(FrozenEstimator(xgb), method="sigmoid").fit(X_val, y_val)
iso   = CalibratedClassifierCV(FrozenEstimator(xgb), method="isotonic").fit(X_val, y_val)

p_platt = platt.predict_proba(X_test)[:, 1]
p_iso   = iso.predict_proba(X_test)[:, 1]

# fixed buckets from the raw prediction, compare all versions against reality
d = pd.DataFrame({"raw": p_raw, "platt": p_platt, "iso": p_iso, "y": y_test.values})
d["bucket"] = pd.qcut(d["raw"], 10, labels=False, duplicates="drop")
tab = d.groupby("bucket").agg(uncalibrated=("raw","mean"), platt=("platt","mean"),
    isotonic=("iso","mean"), really_happened=("y","mean")) * 100
print("All numbers are % (bucketed by raw prediction):\n")
print(tab.round(1).to_string())

print("\nRanking preserved (AUC identical) — calibration only rescales:")
for nm, pp in [("uncalibrated", p_raw), ("platt", p_platt), ("isotonic", p_iso)]:
    print(f"  {nm:12s} AUC = {roc_auc_score(y_test, pp):.4f}")

All numbers are % (bucketed by raw prediction):

        uncalibrated  platt  isotonic  really_happened
bucket                                                
0           6.200000    0.7       0.3              0.5
1           9.300000    0.8       0.4              0.3
2          12.200000    1.0       0.7              1.0
3          15.400000    1.1       0.9              1.1
4          19.700001    1.5       2.0              1.8
5          26.299999    2.1       3.0              2.6
6          36.200001    3.6       4.5              3.6
7          48.400002    6.8       6.9              6.7
8          62.299999   13.6      12.0             12.1
9          85.800003   36.3      36.3             37.1

Ranking preserved (AUC identical) — calibration only rescales:
  uncalibrated AUC = 0.8686
  platt        AUC = 0.8686
  isotonic     AUC = 0.8686


After calibration, `platt` and `isotonic` sit right on top of `really_happened`, and AUC is
unchanged. The model now **means what it says** at no cost to ranking. This is required before any PD
is used to price a loan or set capital (IFRS 9).

## 6. Recap
- All three boosters are close on tabular data; **CatBoost ~= XGBoost** out of the box, **LightGBM**
  needed its own tuning (num_leaves, min_child_samples) to catch up.
- Gradient boosting **ranks** well but is **miscalibrated** (worsened by `scale_pos_weight`).
- **Platt / isotonic** calibration fixes the probabilities without touching the ranking (AUC unchanged).

**Next:** Home Credit — multi-table feature engineering across 7 tables.